In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

Import Acled events data

In [ ]:
PATH_TO_ACLED_EVENTS = '../data/raw/Middle-East_aggregated_data_up_to_week_of-2026-10-31.csv'

df_events = pd.read_csv(PATH_TO_ACLED_EVENTS)

In [ ]:
#df_events.head()

### Preprocessing

In [ ]:
df_events.dtypes

In [ ]:
# strip whitespaces
string_cols = ['REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
df_events[string_cols] = df_events[string_cols].apply(lambda x: x.str.strip())
#df_events[string_cols].head()

In [ ]:
# format date columns
df_events['WEEK'] = pd.to_datetime(df_events['WEEK'])
df_events['WEEK'].head()

In [ ]:
# create date grouping columns
df_events['YEAR'] = df_events['WEEK'].dt.year
df_events['YEARMO'] = df_events['WEEK'].dt.to_period('M').astype(str)
df_events['YEAR_WEEK'] = df_events['WEEK'].dt.to_period('W').astype(str)

<br> 

Filter for Palestine events

In [ ]:
mask_palestine_events = df_events.COUNTRY == 'Palestine'
df_events = df_events[mask_palestine_events]

## EDA

#### **NOTE: Counts

At least for sprint 1, I am choosing to count the number of rows as the number of events for simplicity. This is therefore an underestimate, since the "EVENTS" column in df_events indicates there can be multiple occurrences of a given event at the same location in that week.

#### What geography is covered?

In [ ]:
pd.concat([
    df_events['ADMIN1'].value_counts().rename('count'),
    df_events['ADMIN1'].value_counts(normalize=True).round(2).rename('ratio')
], axis=1)

### Different Event types

In [ ]:
pd.concat([
    df_events['EVENT_TYPE'].value_counts().rename('count'),
    df_events['EVENT_TYPE'].value_counts(normalize=True).round(2).rename('ratio')
], axis=1)

### Different sub event types

In [ ]:
pd.concat([
    df_events['SUB_EVENT_TYPE'].value_counts().rename('count'),
    df_events['SUB_EVENT_TYPE'].value_counts(normalize=True).round(2).rename('ratio')
], axis=1)

### Different disorder types

In [ ]:
pd.concat([
    df_events['DISORDER_TYPE'].value_counts().rename('count'),
    df_events['DISORDER_TYPE'].value_counts(normalize=True).round(2).rename('ratio')
], axis=1)

### Date range

In [ ]:
print(f"Start date: {df_events['WEEK'].min():%Y-%m}")
print(f"End date:   {df_events['WEEK'].max():%Y-%m}")

### Events over time

#### Gaza vs West Bank -- Total Events

This version treats each row as an event (ignoring the fact that there can be multiple instances of that event in the same location, but at different points in the day / week, as indiciated by the 'EVENTS' column in df_events)

In [ ]:
events_year = (
    df_events.groupby(['YEAR', 'ADMIN1'])
    .size()
    .unstack(fill_value=0)
)

ax = events_year.plot(
    kind='line',
    figsize=(12, 5),
    marker='o'
)

ax.set_title('Total Events by Year and Admin1')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Events')
ax.legend(title='Admin1', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticks(events_year.index)

display(events_year)
plt.tight_layout()
plt.show()

#### Gaza vs West Bank -- Total Instances

This version takes into account the fact that there can be multiple instances of each event in the same location, but at different points in the day / week, as indiciated by the 'EVENTS' column in df_events

In [ ]:
events_year = (
    df_events.groupby(['YEAR', 'ADMIN1']).EVENTS
    .sum()
    .unstack(fill_value=0)
)

ax = events_year.plot(
    kind='line',
    figsize=(12, 5),
    marker='o'
)

ax.set_title('Total Events by Year and Admin1')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Events')
ax.legend(title='Admin1', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticks(events_year.index)

display(events_year)
plt.tight_layout()
plt.show()

#### TODO: Seasonality? 

In [ ]:
mask_min_date = df_events.YEAR > 2023

events_month = (
    df_events[mask_min_date].groupby(['YEARMO', 'ADMIN1']).EVENTS
    .sum()
    .unstack(fill_value=0)
)

ax = events_month.plot(
    kind='line',
    figsize=(15, 6),
    marker='o'
)

ax.set_title('Total Events by Month and ADMIN1')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Number of Events')
ax.legend(title='ADMIN1', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
events_week = (
    df_events[mask_min_date].groupby(['YEAR_WEEK', 'ADMIN1']).EVENTS
    .sum()
    .unstack(fill_value=0)
)

ax = events_week.plot(
    kind='line',
    figsize=(16, 6),
    marker='o'
)

ax.set_title('Total Events by Week and ADMIN1')
ax.set_xlabel('Year-Week')
ax.set_ylabel('Number of Events')
ax.legend(title='ADMIN1', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

#### by Event Type

In [ ]:
for admin in df_events.ADMIN1.unique():
    mask_admin = df_events.ADMIN1==admin
    events_by_type = (
        df_events[mask_admin].groupby(['YEAR', 'EVENT_TYPE']).EVENTS
        .sum()
        .unstack(fill_value=0)
    )
    
    ax = events_by_type.plot(
        kind='line',
        figsize=(18, 7)
    )
    
    ax.set_title(f'Events by Event Type: {admin}')
    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Number of Events')
    ax.legend(title='ADMIN1 / Event Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45)
    display(events_by_type)
    plt.tight_layout()
    plt.show()

#### by Sub Event Type

#### TODO: distill from noise

In [ ]:
for admin in df_events.ADMIN1.unique():
    mask_admin = df_events.ADMIN1==admin
    events_by_type = (
        df_events[mask_admin].groupby(['YEAR', 'SUB_EVENT_TYPE']).EVENTS
        .sum()
        .unstack(fill_value=0)
    )
    
    ax = events_by_type.plot(
        kind='line',
        figsize=(18, 7)
    )
    
    ax.set_title(f'Events by Sub Event Type: {admin}')
    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Number of Events')
    ax.legend(title='ADMIN1 / Event Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45)
    display(events_by_type)
    plt.tight_layout()
    plt.show()